In [4]:
import pandas as pd
import numpy as np
import itertools
import datetime
import matplotlib.pyplot as plt
import pandas_gbq
from PIL import Image
from datetime import *
from datetime import datetime, timedelta, date
# %load_ext google.colab.data_table
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
project_id = "perceptive-ivy-290216"

# Standard plotly imports
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

In [5]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
import fastf1
from fastf1 import utils

year=2025
gp="Chinese Grand Prix"
session = fastf1.get_session(2025, 'Chinese Grand Prix', 'SQ')
session.load()

circuit_info = session.get_circuit_info()

core           INFO 	Loading data for Chinese Grand Prix - Sprint Qualifying [v3.5.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core        WARNING 	Sprint Qualifying is not supported by Ergast! Limited results are calculated from timing data.
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['44', '1', '81', '16', '63', '4', '12', '22', '23', '18', '14', '87', '55', '5', '6', '7', '10', '31',

In [ ]:
#Get top laps
results=session.laps
results['Year']=year
results['GP']=gp

Lap_Driver=pd.DataFrame(results.groupby(by=["Driver","LapNumber"])["LapTime"].unique().explode())
Lap_Driver=Lap_Driver.sort_values(by=["Driver","LapTime"])
Lap_Driver["RK"] = Lap_Driver.groupby("Driver")["LapTime"].rank(method="dense", ascending=True)
Lap_Driver.sort_values(by=["LapTime"]).head(10)

In [26]:
driver1="HAM"
driver2="VER"
d1_lap = session.laps.pick_drivers(driver1).pick_fastest()
d2_lap = session.laps.pick_drivers(driver2).pick_fastest()
d1_tel = d1_lap.get_car_data().add_distance()
d2_tel = d2_lap.get_car_data().add_distance()
d1_tel["Driver"]=driver1
d2_tel["Driver"]=driver2


In [32]:
total_tel=pd.concat([d1_tel, d2_tel], axis=0)
total_tel.head()

,Date,RPM,Speed,nGear,Throttle,Brake,DRS,Source,Time,SessionTime,Distance,Driver
0,2025-03-21 08:11:32.438,11138.0,278.0,7,100.0,False,12,car,0 days 00:00:00.130000,0 days 00:54:00.118000,10.038889,HAM
1,2025-03-21 08:11:32.719,11277.0,282.0,7,99.0,False,12,car,0 days 00:00:00.411000,0 days 00:54:00.399000,32.050556,HAM
2,2025-03-21 08:11:32.959,11417.0,285.0,7,99.0,False,12,car,0 days 00:00:00.651000,0 days 00:54:00.639000,51.050556,HAM
3,2025-03-21 08:11:33.159,11508.0,288.0,7,99.0,False,12,car,0 days 00:00:00.851000,0 days 00:54:00.839000,67.050556,HAM
4,2025-03-21 08:11:33.479,11637.0,292.0,7,99.0,False,12,car,0 days 00:00:01.171000,0 days 00:54:01.159000,93.006111,HAM


In [33]:
#Convert braking from True False to 0 & 1
def braking_ind(ind):
  if ind==True:
    return 1
  else:
    return 0

d1_tel['Braking'] = d1_tel['Brake'].apply(braking_ind)
d2_tel['Braking'] = d2_tel['Brake'].apply(braking_ind)
total_tel['Braking'] = total_tel['Brake'].apply(braking_ind)

In [18]:
driver1_delta_lap = session.laps.pick_drivers(driver1).pick_fastest()
driver2_delta_lap = session.laps.pick_drivers(driver2).pick_fastest()


delta_time, ref_tel, compare_tel = utils.delta_time(driver1_delta_lap, driver2_delta_lap)
delta_time=pd.DataFrame(delta_time)
delta_time=delta_time.reset_index()
delta_time.columns=["RK", "Delta"]

compare_tel=compare_tel.reset_index()
compare_tel.columns=["RK", 'Date', 'SessionTime', 'RPM', 'Speed', 'nGear', 'Throttle', 'Brake',
       'DRS', 'Source', 'Time', 'Distance']
compare_tel=compare_tel.merge(delta_time, left_on="RK", right_on="RK")

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/fastf1/utils.py:89: FutureWarning: `utils.delta_time` is considered deprecated and willbe modified or removed in a future release because it hasa tendency to give inaccurate results.
  warnings.warn("`utils.delta_time` is considered deprecated and will"


In [ ]:
v_min = compare_tel['Delta'].min()
v_max = compare_tel['Delta'].max()
corner_distance=circuit_info.corners['Distance']

fig_delta=px.line(data_frame=compare_tel, x="Distance", y="Delta",
           height=450, width=1500,
               template="plotly_dark",
           title="{} {} Sprint Qualifying: Time Delta Between {} vs. {} Fastest Qualifying Laps)".format(year, gp, driver1, driver2),

          )
# fig_speed.add_layout_image(
#     dict(
#         source=Image.open("/content/images.jpg"),
#         xref="x", yref="y",
#         x=4500, y=300,
#         sizex=500, sizey=500,
#         xanchor="left", yanchor="middle"
#     )
# )
fig_delta.update_layout(
    xaxis_title="Distance (m)",
    yaxis_title="Delta (s)",
)

for k in corner_distance:
  fig_delta.add_shape(type='line',
                yref="y",
                xref="x",
                x0=k,
                y0=v_min,
                x1=k,
                y1=v_max,
                line_width=1,
                line_dash="dash",
                line_color="grey")
fig_delta.update_xaxes(range=[0, 5400])
fig_delta.update_yaxes(range=[-0.2, 0.5])

for _, corner in circuit_info.corners.iterrows():
    txt = f"{corner['Number']}{corner['Letter']}"
    fig_delta.add_annotation(x=corner['Distance'], y=v_max, text=txt, showarrow=False, yshift=10)

fig_delta.update_layout(
    title_x=0.5,
    yaxis = dict(tickfont = dict(size=20)),
    xaxis = dict(tickfont = dict(size=20)),
    hoverlabel=dict(
        bgcolor="white",
        font_size=15,
        font_family="PT Sans Narrow"
    ),
    font=dict(
        family="PT Sans Narrow",
        size=18,
        color="White"
    ),
    title_font_family="PT Sans Narrow"

)

fig_delta.update_traces(line_color='White', line_width=2)

fig_delta.show()

In [ ]:
fig_speed=px.line(d1_tel, x="Distance", y="Speed",
           color="Driver", hover_name="Driver",
           height=450, width=1500,
               template="plotly_dark",
           title="{} {} Sprint Qualifying: Speed Trace For {} vs. {} Fastest Qualifying Laps)".format(year, gp, driver1, driver2),
           color_discrete_map={
                 "VER": "#3671C6",
                 "LAW": "#3671C6",
                 "LEC": "#E80020",
                 "HAM": "#E80020",
                 "NOR": "#FF8000",
                 "PIA": "#FF8000",
                 "RUS": "#27F4D2",
                 "ANT": "#27F4D2",
                 "GAS": "#0093CC",
                 "DOO": "#0093CC",
                 "ALO": "#229971",
                 "STR": "#229971",
                 "SAI": "#64C4FF",
                 "ALB": "#64C4FF",
                 "HUL": "#52e252",
                 "BOR": "#52e252",
                 "TSU": "#6692FF",
                 "HAD": "#6692FF",
                 "OCO": "#B6BABD",
                 "BEA": "#B6BABD"
          }
          )
# fig_speed.add_layout_image(
#     dict(
#         source=Image.open("/content/images.jpg"),
#         xref="x", yref="y",
#         x=4500, y=300,
#         sizex=500, sizey=500,
#         xanchor="left", yanchor="middle"
#     )
# )
fig_speed.update_layout(
    xaxis_title="Distance (m)",
    yaxis_title="Speed (Km/Hr)",
)

fig_speed.update_layout(
    title_x=0.5,
    yaxis = dict(tickfont = dict(size=20)),
    xaxis = dict(tickfont = dict(size=20)),
    hoverlabel=dict(
        bgcolor="white",
        font_size=20,
        font_family="PT Sans Narrow"
    ),
    font=dict(
        family="PT Sans Narrow",
        size=24,
        color="White"
    ),
    title_font_family="PT Sans Narrow"
#  "Arial", "Balto", "Courier New", "Droid Sans", "Droid Serif", "Droid Sans Mono", "Gravitas One", "Old Standard TT", "Open Sans", "Overpass", "PT Sans Narrow", "Raleway", "Times New Roman"
)
fig_speed.show()

In [ ]:
fig_speed_addtl=px.line(d2_tel, x="Distance", y="Speed",
           color="Driver", hover_name="Driver",
           height=450, width=1800,
           template="presentation",
           title="{} {} Sprint Qualifying: Speed Trace For {} vs. {} Fastest Qualifying Laps)".format(year, gp, driver1, driver2),
           color_discrete_map={
                 "VER": "#3671C6",
                 "TSU": "#3671C6",
                 "LEC": "#E80020",
                 "HAM": "#E80020",
                 "NOR": "#FF8000",
                 "PIA": "#FF8000",
                 "RUS": "#27F4D2",
                 "ANT": "#27F4D2",
                 "GAS": "#0093CC",
                 "DOO": "#0093CC",
                 "ALO": "#229971",
                 "STR": "#229971",
                 "SAI": "#64C4FF",
                 "ALB": "#64C4FF",
                 "HUL": "#52e252",
                 "BOR": "#52e252",
                 "LAW": "#6692FF",
                 "HAD": "#6692FF",
                 "OCO": "#B6BABD",
                 "BEA": "#B6BABD"
          }
          )

fig_speed.add_traces(
    list(fig_speed_addtl.select_traces())
)

fig_speed.update_layout(
    xaxis_title="Distance (m)",
    yaxis_title="Speed (Km/Hr)",
)
fig_speed.update_layout(
    title_x=0.5,
    hoverlabel=dict(
        bgcolor="white",
        font_size=16,
        font_family="PT Sans Narrow"
    ),
    yaxis = dict(tickfont = dict(size=20)),
    xaxis = dict(tickfont = dict(size=20)),
    font=dict(
        family="PT Sans Narrow",
        size=24,
        color="White"
    ),
    title_font_family="PT Sans Narrow"
#  "Arial", "Balto", "Courier New", "Droid Sans", "Droid Serif", "Droid Sans Mono", "Gravitas One", "Old Standard TT", "Open Sans", "Overpass", "PT Sans Narrow", "Raleway", "Times New Roman"
)

fig_speed.show()

In [163]:
#Speed Trace with Corner Markings

In [23]:
v_min = d1_tel['Speed'].min()
v_max = d1_tel['Speed'].max()
corner_distance=circuit_info.corners['Distance']

for k in corner_distance:
  fig_speed.add_shape(type='line',
                yref="y",
                xref="x",
                x0=k,
                y0=v_min+25,
                x1=k,
                y1=v_max+5,
                line_width=1,
                line_dash="dash",
                line_color="grey")
fig_speed.update_xaxes(range=[0, 5400])
fig_speed.update_yaxes(range=[10, 350])

for _, corner in circuit_info.corners.iterrows():
    txt = f"{corner['Number']}{corner['Letter']}"
    fig_speed.add_annotation(x=corner['Distance'], y=v_max, text=txt, showarrow=False, yshift=10)

fig_speed.update_layout(
   title_x=0.5,
   hoverlabel=dict(
        bgcolor="white",
        font_size=16,
        font_family="PT Sans Narrow"
    ),
    yaxis = dict(tickfont = dict(size=20)),
    xaxis = dict(tickfont = dict(size=20)),
    font=dict(
        family="PT Sans Narrow",
        size=18,
        color="White"
    ),
    title_font_family="PT Sans Narrow"
#  "Arial", "Balto", "Courier New", "Droid Sans", "Droid Serif", "Droid Sans Mono", "Gravitas One", "Old Standard TT", "Open Sans", "Overpass", "PT Sans Narrow", "Raleway", "Times New Roman"
)

fig_speed.show()

In [ ]:
fig_throttle=px.line(total_tel, x="Distance", y="Throttle",
           color="Driver", hover_name="Driver",
           height=900, width=1500,
           template="plotly_dark",
           title="{} {} Sprint Qualifying: Throttle Trace For {} vs. {} Fastest Qualifying Laps)".format(year, gp, driver1, driver2),
           color_discrete_map={
                 "VER": "#3671C6",
                 "TSU": "#3671C6",
                 "LEC": "#E80020",
                 "HAM": "#E80020",
                 "NOR": "#FF8000",
                 "PIA": "#FF8000",
                 "RUS": "#27F4D2",
                 "ANT": "#27F4D2",
                 "GAS": "#0093CC",
                 "DOO": "#0093CC",
                 "ALO": "#229971",
                 "STR": "#229971",
                 "SAI": "#64C4FF",
                 "ALB": "#64C4FF",
                 "HUL": "#52e252",
                 "BOR": "#52e252",
                 "LAW": "#6692FF",
                 "HAD": "#6692FF",
                 "OCO": "#B6BABD",
                 "BEA": "#B6BABD"
          }
          )

fig_throttle.update_layout(
    xaxis_title="Distance (m)",
    yaxis_title="Throttle (%)",
)

v_min = d1_tel['Throttle'].min()
v_max = d1_tel['Throttle'].max()
corner_distance=circuit_info.corners['Distance']

for k in corner_distance:
  fig_throttle.add_shape(type='line',
                yref="y",
                xref="x",
                x0=k,
                y0=v_min-5,
                x1=k,
                y1=v_max+20,
                line_width=1,
                line_dash="dash",
                line_color="grey")
fig_throttle.update_xaxes(range=[0, 5000])
fig_throttle.update_yaxes(range=[-10, 115])

for _, corner in circuit_info.corners.iterrows():
    txt = f"{corner['Number']}{corner['Letter']}"
    fig_throttle.add_annotation(x=corner['Distance'], y=v_max+10, text=txt, showarrow=False, yshift=10)

fig_throttle.update_layout(
   title_x=0.5,
   hoverlabel=dict(
        bgcolor="white",
        font_size=16,
        font_family="PT Sans Narrow"
    ),
    yaxis = dict(tickfont = dict(size=20)),
    xaxis = dict(tickfont = dict(size=20)),
    font=dict(
        family="PT Sans Narrow",
        size=18,
        color="White"
    ),
    title_font_family="PT Sans Narrow"
#  "Arial", "Balto", "Courier New", "Droid Sans", "Droid Serif", "Droid Sans Mono", "Gravitas One", "Old Standard TT", "Open Sans", "Overpass", "PT Sans Narrow", "Raleway", "Times New Roman"
)

fig_throttle.show()

In [ ]:
fig_brake=px.line(total_tel, x="Distance", y="Braking",
           color="Driver", hover_name="Driver",
           height=900, width=1500,
           template="plotly_dark",
           title="{} {} Sprint Qualifying: Brake Trace For {} vs. {} Fastest Qualifying Laps)".format(year, gp, driver1, driver2),
           color_discrete_map={
                 "VER": "#3671C6",
                 "TSU": "#3671C6",
                 "LEC": "#E80020",
                 "HAM": "#E80020",
                 "NOR": "#FF8000",
                 "PIA": "#FF8000",
                 "RUS": "#27F4D2",
                 "ANT": "#27F4D2",
                 "GAS": "#0093CC",
                 "DOO": "#0093CC",
                 "ALO": "#229971",
                 "STR": "#229971",
                 "SAI": "#64C4FF",
                 "ALB": "#64C4FF",
                 "HUL": "#52e252",
                 "BOR": "#52e252",
                 "LAW": "#6692FF",
                 "HAD": "#6692FF",
                 "OCO": "#B6BABD",
                 "BEA": "#B6BABD"
          }
          )

fig_brake.update_layout(
    xaxis_title="Distance (m)",
    yaxis_title="Brakes",
)

v_min = d1_tel['Braking'].min()
v_max = d1_tel['Braking'].max()
corner_distance=circuit_info.corners['Distance']

for k in corner_distance:
  fig_brake.add_shape(type='line',
                yref="y",
                xref="x",
                x0=k,
                y0=v_min-0.25,
                x1=k,
                y1=v_max+0.2,
                line_width=1,
                line_dash="dash",
                line_color="grey")
fig_brake.update_xaxes(range=[0, 5000])
fig_brake.update_yaxes(range=[-0.5, 1.5])

for _, corner in circuit_info.corners.iterrows():
    txt = f"{corner['Number']}{corner['Letter']}"
    fig_brake.add_annotation(x=corner['Distance'], y=v_max+0.2, text=txt, showarrow=False, yshift=10)

fig_brake.update_layout(
   title_x=0.5,
   hoverlabel=dict(
        bgcolor="white",
        font_size=16,
        font_family="PT Sans Narrow"
    ),
    yaxis = dict(tickfont = dict(size=20)),
    xaxis = dict(tickfont = dict(size=20)),
    font=dict(
        family="PT Sans Narrow",
        size=18,
        color="White"
    ),
    title_font_family="PT Sans Narrow"
#  "Arial", "Balto", "Courier New", "Droid Sans", "Droid Serif", "Droid Sans Mono", "Gravitas One", "Old Standard TT", "Open Sans", "Overpass", "PT Sans Narrow", "Raleway", "Times New Roman"
)

fig_brake.show()

In [ ]:
fig_drs=px.line(total_tel, x="Distance", y="DRS",
           color="Driver", hover_name="Driver",
           height=900, width=1500,
           template="plotly_dark",
           title="{} {} Sprint Qualifying: DRS Trace For {} vs. {} Fastest Qualifying Laps)".format(year, gp, driver1, driver2),
           color_discrete_map={
                 "VER": "#3671C6",
                 "TSU": "#3671C6",
                 "LEC": "#E80020",
                 "HAM": "#E80020",
                 "NOR": "#FF8000",
                 "PIA": "#FF8000",
                 "RUS": "#27F4D2",
                 "ANT": "#27F4D2",
                 "GAS": "#0093CC",
                 "DOO": "#0093CC",
                 "ALO": "#229971",
                 "STR": "#229971",
                 "SAI": "#64C4FF",
                 "ALB": "#64C4FF",
                 "HUL": "#52e252",
                 "BOR": "#52e252",
                 "LAW": "#6692FF",
                 "HAD": "#6692FF",
                 "OCO": "#B6BABD",
                 "BEA": "#B6BABD"
          }
          )

fig_drs.update_layout(
    xaxis_title="Distance (m)",
    yaxis_title="DRS",
)

v_min = d1_tel['DRS'].min()
v_max = d1_tel['DRS'].max()
corner_distance=circuit_info.corners['Distance']

for k in corner_distance:
  fig_drs.add_shape(type='line',
                yref="y",
                xref="x",
                x0=k,
                y0=v_min-5,
                x1=k,
                y1=v_max+2,
                line_width=1,
                line_dash="dash",
                line_color="grey")
fig_drs.update_xaxes(range=[0, 5000])
fig_drs.update_yaxes(range=[-0.5, 18])

for _, corner in circuit_info.corners.iterrows():
    txt = f"{corner['Number']}{corner['Letter']}"
    fig_drs.add_annotation(x=corner['Distance'], y=v_min-5.5, text=txt, showarrow=False, yshift=10)
fig_drs.update_layout(
   title_x=0.5,
   hoverlabel=dict(
        bgcolor="white",
        font_size=16,
        font_family="PT Sans Narrow"
    ),
    yaxis = dict(tickfont = dict(size=20)),
    xaxis = dict(tickfont = dict(size=20)),
    font=dict(
        family="PT Sans Narrow",
        size=18,
        color="White"
    ),
    title_font_family="PT Sans Narrow"
#  "Arial", "Balto", "Courier New", "Droid Sans", "Droid Serif", "Droid Sans Mono", "Gravitas One", "Old Standard TT", "Open Sans", "Overpass", "PT Sans Narrow", "Raleway", "Times New Roman"
)
fig_drs.show()